## 用于给apoc和DABOOK做替换并且label迁移的

In [ ]:
from rdkit import Chem
from localmapper import localmapper
from typing import List, Tuple, Dict
import re

def get_atom_positions_from_smiles(mapped_smiles: str) -> Dict[int, int]:
    """
    从SMILES字符串中获取所有原子的位置，包括未映射的原子
    Returns: {smiles_position: mapped_number or None}
    """
    positions = []
    current_pos = 0
    
    # 正则表达式模式
    # 匹配: [X:n] 形式的映射原子，普通原子，以及 [X] 形式的原子
    pattern = r'(\[[^]]+?:(\d+)\])|([A-Z][a-z]?(?!:))|(\[[^]]+?\](?!:))'
    
    for match in re.finditer(pattern, mapped_smiles):
        if match.group(1):  # 匹配到映射原子 [X:n]
            positions.append(int(match.group(2)))  # 存储映射号
        else:  # 未映射原子
            positions.append(None)
        current_pos += 1
    
    return {i: num for i, num in enumerate(positions)}

def get_atom_mapping_from_mapped_smiles(mapped_smiles: str) -> Dict[int, int]:
    """
    从完整的反应SMILES中获取原子映射
    Returns: {source_position: target_position}
    """
    # 分割反应物和产物
    reactant, product = mapped_smiles.split('>>')
    
    # 获取反应物和产物中的原子位置
    reactant_positions = get_atom_positions_from_smiles(reactant)
    product_positions = get_atom_positions_from_smiles(product)
    
    print(f"反应物原子位置: {reactant_positions}")
    print(f"产物原子位置: {product_positions}")
    
    # 创建原始位置到新位置的映射
    mapping = {}
    unmapped_source = []
    unmapped_target = []
    
    # 收集未映射的位置
    for pos, map_num in reactant_positions.items():
        if map_num is None:
            unmapped_source.append(pos)
    
    for pos, map_num in product_positions.items():
        if map_num is None:
            unmapped_target.append(pos)
    
    print(f"未映射的源原子位置: {unmapped_source}")
    print(f"未映射的目标原子位置: {unmapped_target}")
    
    # 首先处理有映射号的原子
    for src_pos, src_map in reactant_positions.items():
        if src_map is not None:  # 有映射号的原子
            for prod_pos, prod_map in product_positions.items():
                if prod_map == src_map:
                    mapping[src_pos] = prod_pos
                    break
    
    # 然后处理未映射的原子（假设它们的顺序相同）
    for src, tgt in zip(unmapped_source, unmapped_target):
        mapping[src] = tgt
    
    print(f"最终映射关系: {mapping}")
    return mapping

def substitute_and_map_atoms(
    smiles: str,
    substituent: str,
    placeholder: str = "[U]",
    atom_pairs: List[Tuple[int, int]] = None,
    device: str = 'cpu'
) -> Tuple[str, List[Tuple[int, int]]]:
    """
    替换SMILES中的占位符并更新原子对的映射
    """
    # 步骤1: 执行替换获取新的SMILES
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string")
    
    rwmol = Chem.RWMol(mol)
    placeholder_mol = Chem.MolFromSmiles(placeholder)
    matches = mol.GetSubstructMatches(placeholder_mol)
    
    if not matches:
        raise ValueError(f"Placeholder {placeholder} not found")
    
    sub_mol = Chem.MolFromSmiles(substituent)
    if sub_mol is None:
        raise ValueError(f"Invalid substituent SMILES: {substituent}")
    
    # 执行替换
    for match in matches:
        placeholder_idx = match[0]
        neighbors = list(mol.GetAtomWithIdx(placeholder_idx).GetNeighbors())
        if not neighbors:
            raise ValueError("Placeholder has no neighbors")
        
        connection_idx = neighbors[0].GetIdx()
        rwmol.RemoveAtom(placeholder_idx)
        
        fragment = Chem.RWMol(sub_mol)
        num_atoms_original = rwmol.GetNumAtoms()
        
        combo = Chem.CombineMols(rwmol, fragment)
        rwmol = Chem.RWMol(combo)
        
        rwmol.AddBond(connection_idx, 
                     num_atoms_original, 
                     Chem.BondType.SINGLE)
    
    # 清理分子
    Chem.SanitizeMol(rwmol)
    new_mol = rwmol.GetMol()
    new_smiles = Chem.MolToSmiles(new_mol)
    
    print(f"原始SMILES: {smiles}")
    print(f"新SMILES: {new_smiles}")
    
    # 使用localmapper获取映射
    reaction_smiles = f"{smiles}>>{new_smiles}"
    mapper = localmapper(device)
    mapped_reaction = mapper.get_atom_map(reaction_smiles)
    print(f"映射的反应SMILES: {mapped_reaction}")
    
    # 获取原子映射
    atom_mapping = get_atom_mapping_from_mapped_smiles(mapped_reaction)
    
    # 更新原子对
    new_atom_pairs = []
    if atom_pairs:
        for pair in atom_pairs:
            if pair[0] in atom_mapping and pair[1] in atom_mapping:
                new_pair = (atom_mapping[pair[0]], atom_mapping[pair[1]])
                new_atom_pairs.append(new_pair)
                print(f"原子对映射: {pair} -> {new_pair}")
    
    return new_smiles, new_atom_pairs

def test_substitution():
    # 测试用例
    smiles = "O=C([U])c1c(-c2ccccc2)ccc2ccccc12"
    atom_pairs = [(3, 4), (11, 12)]
    
    print(f"原始SMILES: {smiles}")
    print(f"原始原子对: {atom_pairs}")
    
    # 测试取代基
    substituents = ["CCC"]  # 简化测试，只使用一个取代基
    
    for sub in substituents:
        try:
            print(f"\n测试取代基: {sub}")
            new_smiles, new_pairs = substitute_and_map_atoms(
                smiles,
                sub,
                "[U]",
                atom_pairs
            )
            
            print(f"结果SMILES: {new_smiles}")
            print(f"新原子对: {new_pairs}")
            
        except Exception as e:
            print(f"错误: {str(e)}")

if __name__ == "__main__":
    test_substitution()

### 改进版（处理多个[U]）

In [ ]:
from rdkit import Chem
from localmapper import localmapper
from typing import List, Tuple, Dict
import re

def get_atom_positions_from_smiles(mapped_smiles: str) -> Dict[int, int]:
    """
    从SMILES字符串中获取所有原子的位置，包括未映射的原子
    Returns: {smiles_position: mapped_number or None}
    """
    positions = []
    current_pos = 0
    
    pattern = r'(\[[^]]+?:(\d+)\])|([A-Z][a-z]?(?!:))|(\[[^]]+?\](?!:))'
    
    for match in re.finditer(pattern, mapped_smiles):
        if match.group(1):  # 匹配到映射原子 [X:n]
            positions.append(int(match.group(2)))  # 存储映射号
        else:  # 未映射原子
            positions.append(None)
        current_pos += 1
    
    return {i: num for i, num in enumerate(positions)}

def get_atom_mapping_from_mapped_smiles(mapped_smiles: str) -> Dict[int, int]:
    """从完整的反应SMILES中获取原子映射"""
    reactant, product = mapped_smiles.split('>>')
    
    reactant_positions = get_atom_positions_from_smiles(reactant)
    product_positions = get_atom_positions_from_smiles(product)
    
    mapping = {}
    unmapped_source = []
    unmapped_target = []
    
    for pos, map_num in reactant_positions.items():
        if map_num is None:
            unmapped_source.append(pos)
    
    for pos, map_num in product_positions.items():
        if map_num is None:
            unmapped_target.append(pos)
    
    for src_pos, src_map in reactant_positions.items():
        if src_map is not None:
            for prod_pos, prod_map in product_positions.items():
                if prod_map == src_map:
                    mapping[src_pos] = prod_pos
                    break
    
    for src, tgt in zip(unmapped_source, unmapped_target):
        mapping[src] = tgt
    
    return mapping

def find_placeholder_positions(mol, placeholder: str = "[U]"):
    """找到所有占位符的位置，按照在SMILES中的顺序返回"""
    placeholder_mol = Chem.MolFromSmiles(placeholder)
    if placeholder_mol is None:
        raise ValueError(f"Invalid placeholder SMILES: {placeholder}")
    
    matches = mol.GetSubstructMatches(placeholder_mol)
    return [match[0] for match in matches]

def substitute_at_position(rwmol, placeholder_idx, substituent, preserve_indices=None):
    """在特定位置进行替换"""
    if preserve_indices is None:
        preserve_indices = set()
    
    neighbors = list(rwmol.GetAtomWithIdx(placeholder_idx).GetNeighbors())
    if not neighbors:
        raise ValueError(f"Placeholder at position {placeholder_idx} has no neighbors")
    
    connection_idx = neighbors[0].GetIdx()
    preserve_indices.add(connection_idx)
    
    sub_mol = Chem.MolFromSmiles(substituent)
    if sub_mol is None:
        raise ValueError(f"Invalid substituent SMILES: {substituent}")
    
    rwmol.RemoveAtom(placeholder_idx)
    preserve_indices = {idx if idx < placeholder_idx else idx - 1 for idx in preserve_indices}
    
    num_atoms_original = rwmol.GetNumAtoms()
    combo = Chem.CombineMols(rwmol, sub_mol)
    rwmol = Chem.RWMol(combo)
    
    rwmol.AddBond(connection_idx, num_atoms_original, Chem.BondType.SINGLE)
    
    return rwmol, preserve_indices

def perform_substitutions(
    smiles: str,
    substitutions: List[Tuple[str, str]],  # [(sub1_for_U1, sub2_for_U2), ...]
    placeholder: str = "[U]",
    atom_pairs: List[Tuple[int, int]] = None,
    device: str = 'cpu'
) -> List[Tuple[str, List[Tuple[int, int]], Tuple[str, str]]]:
    """
    在SMILES中的特定位置替换[U]并更新原子对的映射
    substitutions: 每个元组包含两个字符串，分别表示第一个和第二个[U]的替换基团
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string")
    
    placeholder_positions = find_placeholder_positions(mol, placeholder)
    if len(placeholder_positions) != 2:
        raise ValueError(f"Expected exactly 2 {placeholder} in SMILES, found {len(placeholder_positions)}")
    
    results = []
    
    for sub1, sub2 in substitutions:
        try:
            rwmol = Chem.RWMol(mol)
            preserve_indices = set()
            
            if atom_pairs:
                preserve_indices.update([idx for pair in atom_pairs for idx in pair])
            
            # 从后向前替换以避免索引问题
            # 首先替换第二个[U]
            rwmol, preserve_indices = substitute_at_position(
                rwmol, 
                placeholder_positions[1], 
                sub2, 
                preserve_indices
            )
            
            # 然后替换第一个[U]
            rwmol, preserve_indices = substitute_at_position(
                rwmol, 
                placeholder_positions[0], 
                sub1, 
                preserve_indices
            )
            
            Chem.SanitizeMol(rwmol)
            new_mol = rwmol.GetMol()
            new_smiles = Chem.MolToSmiles(new_mol)
            
            # 使用localmapper获取映射
            reaction_smiles = f"{smiles}>>{new_smiles}"
            mapper = localmapper(device)
            mapped_reaction = mapper.get_atom_map(reaction_smiles)
            
            # 获取原子映射
            atom_mapping = get_atom_mapping_from_mapped_smiles(mapped_reaction)
            
            # 更新原子对
            new_atom_pairs = []
            if atom_pairs:
                for pair in atom_pairs:
                    if pair[0] in atom_mapping and pair[1] in atom_mapping:
                        new_pair = (atom_mapping[pair[0]], atom_mapping[pair[1]])
                        new_atom_pairs.append(new_pair)
            
            results.append((new_smiles, new_atom_pairs, (sub1, sub2)))
            
        except Exception as e:
            print(f"Error with substitution {(sub1, sub2)}: {str(e)}")
            continue
    
    return results

def test_substitution():
    # 测试用例
    smiles = "C(/[U])(\CCC)=C(/CCC)\[U]"
    atom_pairs = [(0, 5)]
    
    # 指定需要的替换组合
    substitutions = [
        ("C", "C"),    # 第一个[U]->C, 第二个[U]->C
        ("C", "CC"),   # 第一个[U]->C, 第二个[U]->CC
        ("C", "CCC"),  # 第一个[U]->C, 第二个[U]->CCC
        ("CC", "C"),   # 第一个[U]->CC, 第二个[U]->C
        ("CCC", "C"),  # 第一个[U]->CCC, 第二个[U]->C
        ("CCC", "CC"), # 第一个[U]->CCC, 第二个[U]->CC
        ("CC", "CCC")  # 第一个[U]->CC, 第二个[U]->CCC
    ]
    
    print(f"原始SMILES: {smiles}")
    print(f"原始原子对: {atom_pairs}")
    print(f"替换组合: {substitutions}")
    
    try:
        results = perform_substitutions(
            smiles,
            substitutions,
            "[U]",
            atom_pairs
        )
        
        print(f"\n共生成 {len(results)} 个有效替换结果:")
        for i, (new_smiles, new_pairs, subs) in enumerate(results, 1):
            print(f"\n结果 {i}:")
            print(f"替换方案: 第一个[U]->{subs[0]}, 第二个[U]->{subs[1]}")
            print(f"SMILES: {new_smiles}")
            print(f"原子对: {new_pairs}")
            
    except Exception as e:
        print(f"错误: {str(e)}")

if __name__ == "__main__":
    test_substitution()

In [ ]:
# 整合后的代码
import ast
from rdkit import Chem
from localmapper import localmapper
import pandas as pd
from typing import List, Tuple

def substitute_and_map_atoms(
    smiles: str,
    substituent: str,
    placeholder: str = "[U]",
    atom_pairs: List[Tuple[int, int]] = None,
    device: str = 'cpu'
) -> Tuple[str, List[Tuple[int, int]]]:
    """
    替换SMILES中的占位符并更新原子对的映射
    """
    # 步骤1: 执行替换获取新的SMILES
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string")
    
    rwmol = Chem.RWMol(mol)
    placeholder_mol = Chem.MolFromSmiles(placeholder)
    matches = mol.GetSubstructMatches(placeholder_mol)
    
    if not matches:
        raise ValueError(f"Placeholder {placeholder} not found")
    
    sub_mol = Chem.MolFromSmiles(substituent)
    if sub_mol is None:
        raise ValueError(f"Invalid substituent SMILES: {substituent}")
    
    # 执行替换
    for match in matches:
        placeholder_idx = match[0]
        neighbors = list(mol.GetAtomWithIdx(placeholder_idx).GetNeighbors())
        if not neighbors:
            raise ValueError("Placeholder has no neighbors")
        
        connection_idx = neighbors[0].GetIdx()
        rwmol.RemoveAtom(placeholder_idx)
        
        fragment = Chem.RWMol(sub_mol)
        num_atoms_original = rwmol.GetNumAtoms()
        
        combo = Chem.CombineMols(rwmol, fragment)
        rwmol = Chem.RWMol(combo)
        
        rwmol.AddBond(connection_idx, 
                     num_atoms_original, 
                     Chem.BondType.SINGLE)
    
    # 清理分子
    Chem.SanitizeMol(rwmol)
    new_mol = rwmol.GetMol()
    new_smiles = Chem.MolToSmiles(new_mol)
    
    print(f"原始SMILES: {smiles}")
    print(f"新SMILES: {new_smiles}")
    
    # 使用localmapper获取映射
    reaction_smiles = f"{smiles}>>{new_smiles}"
    mapper = localmapper(device)
    mapped_reaction = mapper.get_atom_map(reaction_smiles)
    print(f"映射的反应SMILES: {mapped_reaction}")
    
    # 获取原子映射
    atom_mapping = get_atom_mapping_from_mapped_smiles(mapped_reaction)
    
    # 更新原子对
    new_atom_pairs = []
    if atom_pairs:
        for pair in atom_pairs:
            if pair[0] in atom_mapping and pair[1] in atom_mapping:
                new_pair = (atom_mapping[pair[0]], atom_mapping[pair[1]])
                new_atom_pairs.append(new_pair)
                print(f"原子对映射: {pair} -> {new_pair}")
    
    return new_smiles, new_atom_pairs

def perform_substitutions(
    smiles: str,
    substitutions: List[Tuple[str, str]],  # [(sub1_for_U1, sub2_for_U2), ...]
    placeholder: str = "[U]",
    atom_pairs: List[Tuple[int, int]] = None,
    device: str = 'cpu'
) -> List[Tuple[str, List[Tuple[int, int]], Tuple[str, str]]]:
    """
    在SMILES中的特定位置替换[U]并更新原子对的映射
    substitutions: 每个元组包含两个字符串，分别表示第一个和第二个[U]的替换基团
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string")
    
    placeholder_positions = find_placeholder_positions(mol, placeholder)
    if len(placeholder_positions) != 2:
        raise ValueError(f"Expected exactly 2 {placeholder} in SMILES, found {len(placeholder_positions)}")
    
    results = []
    
    for sub1, sub2 in substitutions:
        try:
            rwmol = Chem.RWMol(mol)
            preserve_indices = set()
            
            if atom_pairs:
                preserve_indices.update([idx for pair in atom_pairs for idx in pair])
            
            # 从后向前替换以避免索引问题
            # 首先替换第二个[U]
            rwmol, preserve_indices = substitute_at_position(
                rwmol, 
                placeholder_positions[1], 
                sub2, 
                preserve_indices
            )
            
            # 然后替换第一个[U]
            rwmol, preserve_indices = substitute_at_position(
                rwmol, 
                placeholder_positions[0], 
                sub1, 
                preserve_indices
            )
            
            Chem.SanitizeMol(rwmol)
            new_mol = rwmol.GetMol()
            new_smiles = Chem.MolToSmiles(new_mol)
            
            # 使用localmapper获取映射
            reaction_smiles = f"{smiles}>>{new_smiles}"
            mapper = localmapper(device)
            mapped_reaction = mapper.get_atom_map(reaction_smiles)
            
            # 获取原子映射
            atom_mapping = get_atom_mapping_from_mapped_smiles(mapped_reaction)
            
            # 更新原子对
            new_atom_pairs = []
            if atom_pairs:
                for pair in atom_pairs:
                    if pair[0] in atom_mapping and pair[1] in atom_mapping:
                        new_pair = (atom_mapping[pair[0]], atom_mapping[pair[1]])
                        new_atom_pairs.append(new_pair)
            
            results.append((new_smiles, new_atom_pairs, (sub1, sub2)))
            
        except Exception as e:
            print(f"Error with substitution {(sub1, sub2)}: {str(e)}")
            continue
    
    return results


def get_atom_mapping_from_mapped_smiles(mapped_smiles: str) -> Dict[int, int]:
    """从完整的反应SMILES中获取原子映射"""
    reactant, product = mapped_smiles.split('>>')
    
    reactant_positions = get_atom_positions_from_smiles(reactant)
    product_positions = get_atom_positions_from_smiles(product)
    
    mapping = {}
    unmapped_source = []
    unmapped_target = []
    
    for pos, map_num in reactant_positions.items():
        if map_num is None:
            unmapped_source.append(pos)
    
    for pos, map_num in product_positions.items():
        if map_num is None:
            unmapped_target.append(pos)
    
    for src_pos, src_map in reactant_positions.items():
        if src_map is not None:
            for prod_pos, prod_map in product_positions.items():
                if prod_map == src_map:
                    mapping[src_pos] = prod_pos
                    break
    
    for src, tgt in zip(unmapped_source, unmapped_target):
        mapping[src] = tgt
    
    return mapping

def count_placeholders(smiles: str) -> int:
    """计算SMILES中[U]的数量"""
    return smiles.count("[U]")

def process_smiles_entry(
    smiles: str,
    marked_bonds: List[Tuple[int, int]],
    single_substituents: List[str] = ["C", "CC", "CCC"],
    double_substituents: List[Tuple[str, str]] = [
        ("C", "C"),    
        ("C", "CC"),   
        ("C", "CCC"),  
        ("CC", "C"),   
        ("CCC", "C"),  
        ("CCC", "CC"), 
        ("CC", "CCC")  
    ]
) -> List[Tuple[str, List[Tuple[int, int]], Tuple[str, ...]]]:
    """处理单个SMILES条目"""
    num_placeholders = count_placeholders(smiles)
    results = []
    
    try:
        if num_placeholders == 1:
            # 处理单个[U]的情况
            for sub in single_substituents:
                try:
                    new_smiles, new_pairs = substitute_and_map_atoms(
                        smiles,
                        sub,
                        "[U]",
                        marked_bonds
                    )
                    results.append((new_smiles, new_pairs, (sub,)))
                except Exception as e:
                    print(f"Error processing single substitution {sub} for {smiles}: {str(e)}")
                    
        elif num_placeholders == 2:
            # 处理两个[U]的情况
            results = perform_substitutions(
                smiles,
                double_substituents,
                "[U]",
                marked_bonds
            )
            
    except Exception as e:
        print(f"Error processing SMILES {smiles}: {str(e)}")
        
    return results

def process_csv_file(file_path: str) -> pd.DataFrame:
    """处理CSV文件并生成新的数据框"""
    # 读取CSV文件
    df = pd.read_csv(file_path)
    
    # 准备结果列表
    results = []
    
    # 处理每一行
    for idx, row in df.iterrows():
        try:
            smiles = row['SMILES']
            
            # 检查是否包含[U]
            if "[U]" not in smiles:
                continue
            
            # 尝试解析marked_bonds
            marked_bonds = ast.literal_eval(row['Marked Bonds'])
            
            # 尝试处理替换
            substitution_results = process_smiles_entry(smiles, marked_bonds)
            
            # 添加结果
            for new_smiles, new_pairs, subs in substitution_results:
                results.append({
                    'Original_SMILES': smiles,
                    'New_SMILES': new_smiles,
                    'Original_Bonds': row['Marked Bonds'],
                    'New_Bonds': str(new_pairs),
                    'Substitutions': str(subs)
                })
                
        except Exception as e:
            # 仅打印原始数据并继续
            print(f"跳过处理的行:")
            print(f"Original_SMILES: {row['SMILES']}")
            print(f"Original_Bonds: {row['Marked Bonds']}")
            continue
    
    # 创建新的数据框
    return pd.DataFrame(results)

def main():
    # 设置输入输出文件路径
    input_file = "marked_bonds_apoc.csv"
    output_file = "processed_smiles_apoc.csv"
    
    # 处理文件
    try:
        results_df = process_csv_file(input_file)
        
        # 保存结果
        results_df.to_csv(output_file, index=False)
        print(f"处理完成。结果已保存到 {output_file}")
        print(f"共处理 {len(results_df)} 个替换结果")
        
    except Exception as e:
        print(f"处理文件时发生错误: {str(e)}")

if __name__ == "__main__":
    main()

## 用于标准化NCE中[H]并且label迁移的

In [ ]:
from rdkit import Chem
from localmapper import localmapper
from typing import List, Tuple, Dict
import re

def adjust_labels_for_hydrogens(smiles: str, atom_pairs: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """根据[H]调整原子标签"""
    def count_hydrogens_before(pos: int) -> int:
        """计算给定位置之前的[H]数量"""
        smiles_prefix = smiles[:pos]
        h_pattern = r'\[H\]'
        return len(re.findall(h_pattern, smiles_prefix))
    
    adjusted_pairs = []
    for start, end in atom_pairs:
        h_before_start = count_hydrogens_before(start)
        h_before_end = count_hydrogens_before(end)
        
        new_start = start - h_before_start
        new_end = end - h_before_end
        adjusted_pairs.append((new_start, new_end))
        
        print(f"原始对: ({start}, {end})")
        print(f"起始点前的[H]数: {h_before_start}")
        print(f"终点前的[H]数: {h_before_end}")
        print(f"调整后的对: ({new_start}, {new_end})")
    
    return adjusted_pairs

def get_atom_mapping(mapped_smiles: str) -> Dict[int, int]:
    """从映射的SMILES中获取原子映射"""
    positions = {}
    pos = 0
    for match in re.finditer(r'\[.+?:(\d+)\]|[A-Z][a-z]?(?!:)', mapped_smiles):
        map_num = match.group(1)
        if map_num:
            positions[pos] = int(map_num)
        pos += 1
    return positions

def standardize_smiles_with_mapping(
    smiles: str,
    atom_pairs: List[Tuple[int, int]],
    device: str = 'cpu'
) -> Tuple[str, List[Tuple[int, int]]]:
    """
    标准化SMILES并更新原子对标记
    """
    print(f"原始SMILES: {smiles}")
    print(f"原始原子对: {atom_pairs}")
    
    # 调整原子对以考虑[H]
    adjusted_pairs = adjust_labels_for_hydrogens(smiles, atom_pairs)
    print(f"考虑[H]后的原子对: {adjusted_pairs}")
    
    # 获取标准化SMILES
    mol = Chem.MolFromSmiles(smiles)
    canonical_smiles = Chem.MolToSmiles(mol)
    print(f"标准化SMILES: {canonical_smiles}")
    
    # 获取原子映射
    reaction_smiles = f"{smiles}>>{canonical_smiles}"
    mapper = localmapper(device)
    mapped_reaction = mapper.get_atom_map(reaction_smiles)
    print(f"映射的反应SMILES: {mapped_reaction}")
    
    # 解析映射的SMILES
    reactant, product = mapped_reaction.split('>>')
    reactant_map = get_atom_mapping(reactant)
    product_map = get_atom_mapping(product)
    
    print(f"反应物映射: {reactant_map}")
    print(f"产物映射: {product_map}")
    
    # 创建位置映射字典
    pos_mapping = {}
    for old_pos, map_num in reactant_map.items():
        for new_pos, new_map in product_map.items():
            if new_map == map_num:
                pos_mapping[old_pos] = new_pos
                break
    
    print(f"位置映射关系: {pos_mapping}")
    
    # 更新调整后的原子对
    new_atom_pairs = []
    for pair in adjusted_pairs:
        if pair[0] in pos_mapping and pair[1] in pos_mapping:
            new_pair = (pos_mapping[pair[0]], pos_mapping[pair[1]])
            new_atom_pairs.append(new_pair)
            print(f"原子对映射: {pair} -> {new_pair}")
    
    return canonical_smiles, new_atom_pairs

def test():
    # 测试用例
    smiles = "[H][C@@]12CCC3=CC(=O)CC[C@]3([H])[C@@]1([H])CC[C@]1(C)[C@@](OC(C)=O)(C(C)=O)C(=C)C[C@@]21[H]"
    atom_pairs = [(4, 26)]
    
    try:
        new_smiles, new_pairs = standardize_smiles_with_mapping(smiles, atom_pairs)
        print(f"\n最终结果:")
        print(f"标准化的SMILES: {new_smiles}")
        print(f"新的原子对: {new_pairs}")
        
        # 验证结果
        new_mol = Chem.MolFromSmiles(new_smiles)
        if new_mol is None:
            print("警告：生成的SMILES无效！")
        else:
            print(f"分子包含 {new_mol.GetNumAtoms()} 个原子")
            for pair in new_pairs:
                if max(pair) >= new_mol.GetNumAtoms():
                    print(f"警告：原子对 {pair} 超出分子原子数量范围！")
                
    except Exception as e:
        print(f"错误: {str(e)}")

if __name__ == "__main__":
    test()

## 师姐name_reaction

In [2]:
import itertools
from rdkit import Chem

def generate_r_groups():
    """Generate R group replacements"""
    return ['C', 'CC', 'CCC']

def generate_ar_groups():
    """Generate Ar group replacements"""
    return ['c1ccccc1', 'c1ccc(C)cc1']  # 苯基和对甲苯基

def generate_y_groups():
    """Generate Y group replacements"""
    return ['O', 'S', 'N']  # NH will be handled separately for SMILES compatibility

def generate_ewg_groups():
    """Generate EWG group replacements"""
    return ['CN', 'COCC', 'C(=O)OCC']  # CN, COEt, COOEt

def generate_x_groups():
    """Generate X group replacements"""
    return ['Cl', 'Br']

def find_pattern_groups(smiles, pattern):
    """Find all unique pattern groups in the SMILES string"""
    import re
    patterns = re.findall(f'\\[{pattern}\\d*\\]', smiles)
    return sorted(set(patterns))

def is_catalyst_or_condition(molecule):
    """Check if a molecule string represents a catalyst or condition"""
    catalyst_patterns = ['[Pd', '[Pt', '[Rh', '[Ru', '[Fe', '[Cu', '[Ni', '[Au', '[Ag']
    if molecule.startswith('[') and molecule.endswith(']'):
        return any(molecule.startswith(pat) for pat in catalyst_patterns)
    return False

def simplify_multistep_reaction(reaction_smiles):
    """Simplify a multistep reaction to single step"""
    steps = reaction_smiles.split('>')
    all_reactants = []
    
    for step in steps[:-1]:
        if step:
            molecules = step.split('.')
            for mol in molecules:
                if mol and not is_catalyst_or_condition(mol):
                    all_reactants.append(mol)
    
    final_products = steps[-1]
    seen = set()
    unique_reactants = []
    for r in all_reactants:
        if r not in seen:
            seen.add(r)
            unique_reactants.append(r)
    
    return f"{'.'.join(unique_reactants)}>>{final_products}"

def is_symmetric_equivalent(smiles1, smiles2):
    """Check if two SMILES strings represent the same molecule considering symmetry"""
    try:
        mol1 = Chem.MolFromSmiles(smiles1)
        mol2 = Chem.MolFromSmiles(smiles2)
        if mol1 is None or mol2 is None:
            return False
        return Chem.MolToInchiKey(mol1) == Chem.MolToInchiKey(mol2)
    except:
        return False

def process_x_replacements(smiles):
    """Process X group replacements including X2 and X3"""
    for i in range(3, 0, -1):  # Handle X3, X2, X1 in order
        pattern = f'[X{i}]' if i > 1 else '[X]'
        replacement = [f'[{x*i}]' for x in generate_x_groups()]
        if pattern in smiles:
            results = []
            for x_val in replacement:
                results.append(smiles.replace(pattern, x_val))
            return results
    return [smiles]

def convert_reaction_smiles(smiles):
    """Convert a generic reaction SMILES to specific reactions with all possible combinations"""
    smiles = simplify_multistep_reaction(smiles)
    reactants, products = smiles.split('>>')
    
    # Find all pattern groups
    r_patterns = find_pattern_groups(smiles, 'R')
    ar_patterns = find_pattern_groups(smiles, 'Ar')
    y_patterns = find_pattern_groups(smiles, 'Y')
    ewg_patterns = find_pattern_groups(smiles, 'EWG')
    
    # Generate all possible combinations
    r_groups = generate_r_groups()
    ar_groups = generate_ar_groups()
    y_groups = generate_y_groups()
    ewg_groups = generate_ewg_groups()
    
    r_combinations = list(itertools.product(r_groups, repeat=len(r_patterns)))
    ar_combinations = list(itertools.product(ar_groups, repeat=len(ar_patterns)))
    y_combinations = list(itertools.product(y_groups, repeat=len(y_patterns)))
    ewg_combinations = list(itertools.product(ewg_groups, repeat=len(ewg_patterns)))
    
    unique_reactions = []
    
    # Generate all combinations
    all_combinations = itertools.product(
        r_combinations,
        ar_combinations,
        y_combinations,
        ewg_combinations
    )
    
    for combo in all_combinations:
        r_combo, ar_combo, y_combo, ewg_combo = combo
        current_reactants = reactants
        current_products = products
        
        # Replace R groups
        for r_pattern, r_value in zip(r_patterns, r_combo):
            current_reactants = current_reactants.replace(r_pattern, r_value)
            current_products = current_products.replace(r_pattern, r_value)
        
        # Replace Ar groups
        for ar_pattern, ar_value in zip(ar_patterns, ar_combo):
            current_reactants = current_reactants.replace(ar_pattern, ar_value)
            current_products = current_products.replace(ar_pattern, ar_value)
        
        # Replace Y groups (handle NH special case)
        for y_pattern, y_value in zip(y_patterns, y_combo):
            y_replacement = '[NH]' if y_value == 'N' else f'[{y_value}]'
            current_reactants = current_reactants.replace(y_pattern, y_replacement)
            current_products = current_products.replace(y_pattern, y_replacement)
        
        # Replace EWG groups
        for ewg_pattern, ewg_value in zip(ewg_patterns, ewg_combo):
            current_reactants = current_reactants.replace(ewg_pattern, ewg_value)
            current_products = current_products.replace(ewg_pattern, ewg_value)
        
        # Process X replacements
        x_variants = process_x_replacements(current_reactants)
        for x_variant in x_variants:
            x_products = process_x_replacements(current_products)[0]  # Assume product follows same X pattern
            full_reaction = f"{x_variant}>>{x_products}"
            
            # Check for symmetry equivalence
            is_unique = True
            for existing_reaction in unique_reactions:
                existing_reactants, existing_products = existing_reaction.split('>>')
                if (is_symmetric_equivalent(x_variant, existing_reactants) and 
                    is_symmetric_equivalent(x_products, existing_products)):
                    is_unique = False
                    break
            
            if is_unique:
                unique_reactions.append(full_reaction)
    
    return unique_reactions

# Test with a reaction containing multiple patterns
test_reaction = "[Ar]O>S=C(Cl)[F,Cl,Br,I][R]>[Ar]S"
converted = convert_reaction_smiles(test_reaction)

# Print all generated unique reactions
for idx, rxn in enumerate(converted, 1):
    print(f"Reaction {idx}: {rxn}")
    
print(f"\nTotal unique reactions generated: {len(converted)}")

Reaction 1: c1ccccc1O.S=C(Cl)[F,Cl,Br,I]C>>c1ccccc1S
Reaction 2: c1ccc(C)cc1O.S=C(Cl)[F,Cl,Br,I]C>>c1ccc(C)cc1S
Reaction 3: c1ccccc1O.S=C(Cl)[F,Cl,Br,I]CC>>c1ccccc1S
Reaction 4: c1ccc(C)cc1O.S=C(Cl)[F,Cl,Br,I]CC>>c1ccc(C)cc1S
Reaction 5: c1ccccc1O.S=C(Cl)[F,Cl,Br,I]CCC>>c1ccccc1S
Reaction 6: c1ccc(C)cc1O.S=C(Cl)[F,Cl,Br,I]CCC>>c1ccc(C)cc1S

Total unique reactions generated: 6


In [ ]:
import itertools
from rdkit import Chem
import pandas as pd
import traceback
from tqdm import tqdm
from datetime import datetime

def generate_r_groups():
    """Generate R group replacements"""
    return ['C', 'CC', 'CCC']

def generate_ar_groups():
    """Generate Ar group replacements"""
    return ['c1ccccc1', 'c1ccc(C)cc1']  # 苯基和对甲苯基

def generate_y_groups():
    """Generate Y group replacements"""
    return ['O', 'S', 'N']  # NH will be handled separately for SMILES compatibility

def generate_ewg_groups():
    """Generate EWG group replacements"""
    return ['CN', 'COCC', 'C(=O)OCC']  # CN, COEt, COOEt

def generate_x_groups():
    """Generate X group replacements"""
    return ['Cl', 'Br']

def find_pattern_groups(smiles, pattern):
    """Find all unique pattern groups in the SMILES string"""
    import re
    patterns = re.findall(f'\\[{pattern}\\d*\\]', smiles)
    return sorted(set(patterns))

def is_catalyst_or_condition(molecule):
    """Check if a molecule string represents a catalyst or condition"""
    catalyst_patterns = ['[Pd', '[Pt', '[Rh', '[Ru', '[Fe', '[Cu', '[Ni', '[Au', '[Ag', '[Li']
    if molecule.startswith('[') and molecule.endswith(']'):
        return any(molecule.startswith(pat) for pat in catalyst_patterns)
    return False

def simplify_multistep_reaction(reaction_smiles):
    """Simplify a multistep reaction to single step"""
    steps = reaction_smiles.split('>')
    all_reactants = []
    
    for step in steps[:-1]:
        if step:
            molecules = step.split('.')
            for mol in molecules:
                if mol and not is_catalyst_or_condition(mol):
                    all_reactants.append(mol)
    
    final_products = steps[-1]
    seen = set()
    unique_reactants = []
    for r in all_reactants:
        if r not in seen:
            seen.add(r)
            unique_reactants.append(r)
    
    return f"{'.'.join(unique_reactants)}>>{final_products}"

def is_symmetric_equivalent(smiles1, smiles2):
    """Check if two SMILES strings represent the same molecule considering symmetry"""
    try:
        mol1 = Chem.MolFromSmiles(smiles1)
        mol2 = Chem.MolFromSmiles(smiles2)
        if mol1 is None or mol2 is None:
            return False
        return Chem.MolToInchiKey(mol1) == Chem.MolToInchiKey(mol2)
    except:
        return False

def process_x_replacements(smiles):
    """Process X group replacements including X2 and X3"""
    for i in range(3, 0, -1):  # Handle X3, X2, X1 in order
        pattern = f'[X{i}]' if i > 1 else '[X]'
        replacement = [f'[{x*i}]' for x in generate_x_groups()]
        if pattern in smiles:
            results = []
            for x_val in replacement:
                results.append(smiles.replace(pattern, x_val))
            return results
    return [smiles]

def process_halogen_replacements(smiles):
    """Process [F, Cl, Br, I] pattern into separate replacements"""
    # Match the exact pattern including spaces
    pattern = '[F, Cl, Br, I]'
    if pattern in smiles:
        results = []
        for halogen in ['F', 'Cl', 'Br', 'I']:
            # Create a new variant with this halogen
            new_smiles = smiles.replace(pattern, f'[{halogen}]')
            results.append(new_smiles)
        return results
    # Try without spaces pattern
    pattern_no_spaces = '[F,Cl,Br,I]'
    if pattern_no_spaces in smiles:
        results = []
        for halogen in ['F', 'Cl', 'Br', 'I']:
            # Create a new variant with this halogen
            new_smiles = smiles.replace(pattern_no_spaces, f'[{halogen}]')
            results.append(new_smiles)
        return results
    return [smiles]

def convert_reaction_smiles(smiles):
    """Convert a generic reaction SMILES to specific reactions with all possible combinations"""
    smiles = simplify_multistep_reaction(smiles)
    reactants, products = smiles.split('>>')
    
    # Find all pattern groups
    r_patterns = find_pattern_groups(smiles, 'R')
    ar_patterns = find_pattern_groups(smiles, 'Ar')
    y_patterns = find_pattern_groups(smiles, 'Y')
    ewg_patterns = find_pattern_groups(smiles, 'EWG')
    
    # Generate all possible combinations
    r_groups = generate_r_groups()
    ar_groups = generate_ar_groups()
    y_groups = generate_y_groups()
    ewg_groups = generate_ewg_groups()
    halogens = ['F', 'Cl', 'Br', 'I']  # Define halogens list
    
    r_combinations = list(itertools.product(r_groups, repeat=len(r_patterns)))
    ar_combinations = list(itertools.product(ar_groups, repeat=len(ar_patterns)))
    y_combinations = list(itertools.product(y_groups, repeat=len(y_patterns)))
    ewg_combinations = list(itertools.product(ewg_groups, repeat=len(ewg_patterns)))
    
    unique_reactions = []
    
    # Generate all combinations including halogens
    all_combinations = itertools.product(
        r_combinations,
        ar_combinations,
        y_combinations,
        ewg_combinations,
        halogens  # Add halogens to the product
    )
    
    for combo in all_combinations:
        r_combo, ar_combo, y_combo, ewg_combo, halogen = combo  # Unpack including halogen
        current_reactants = reactants
        current_products = products
        
        # Replace halogen pattern
        halogen_pattern = '[F,Cl,Br,I]'
        alt_halogen_pattern = '[F, Cl, Br, I]'
        
        if halogen_pattern in current_reactants:
            current_reactants = current_reactants.replace(halogen_pattern, f'[{halogen}]')
        if halogen_pattern in current_products:
            current_products = current_products.replace(halogen_pattern, f'[{halogen}]')
            
        if alt_halogen_pattern in current_reactants:
            current_reactants = current_reactants.replace(alt_halogen_pattern, f'[{halogen}]')
        if alt_halogen_pattern in current_products:
            current_products = current_products.replace(alt_halogen_pattern, f'[{halogen}]')
        
        # Replace R groups
        for r_pattern, r_value in zip(r_patterns, r_combo):
            current_reactants = current_reactants.replace(r_pattern, r_value)
            current_products = current_products.replace(r_pattern, r_value)
        
        # Replace Ar groups
        for ar_pattern, ar_value in zip(ar_patterns, ar_combo):
            current_reactants = current_reactants.replace(ar_pattern, ar_value)
            current_products = current_products.replace(ar_pattern, ar_value)
        
        # Replace Y groups (handle NH special case)
        for y_pattern, y_value in zip(y_patterns, y_combo):
            y_replacement = '[NH]' if y_value == 'N' else f'[{y_value}]'
            current_reactants = current_reactants.replace(y_pattern, y_replacement)
            current_products = current_products.replace(y_pattern, y_replacement)
        
        # Replace EWG groups
        for ewg_pattern, ewg_value in zip(ewg_patterns, ewg_combo):
            current_reactants = current_reactants.replace(ewg_pattern, ewg_value)
            current_products = current_products.replace(ewg_pattern, ewg_value)
        
        # Process X replacements if present
        x_variants = process_x_replacements(current_reactants)
        for x_variant in x_variants:
            p_x_variant = process_x_replacements(current_products)[0]
            full_reaction = f"{x_variant}>>{p_x_variant}"
            
            # Check for symmetry equivalence
            is_unique = True
            for existing_reaction in unique_reactions:
                existing_reactants, existing_products = existing_reaction.split('>>')
                if (is_symmetric_equivalent(x_variant, existing_reactants) and 
                    is_symmetric_equivalent(p_x_variant, existing_products)):
                    is_unique = False
                    break
            
            if is_unique:
                unique_reactions.append(full_reaction)
    
    return unique_reactions


def process_reactions_from_csv():
    """
    Process reactions from CSV file and save results
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    try:
        # Read input file without header, first column is SMILES
        df = pd.read_csv('name_reaction_smiles.csv', header=None)
        print(f"Successfully loaded CSV file with {len(df)} reactions")
    except Exception as e:
        print(f"Error reading input file: {str(e)}")
        return

    # Initialize results storage
    successful_reactions = []
    failed_reactions = []
    
    # Process each reaction
    for idx, row in tqdm(df.iterrows(), desc="Processing reactions"):
        try:
            original_smiles = str(row[0])  # Get SMILES from first column
            
            # Skip empty or invalid SMILES
            if pd.isna(original_smiles) or not original_smiles.strip():
                raise ValueError("Empty or invalid SMILES string")
            
            # Convert reaction
            converted_reactions = convert_reaction_smiles(original_smiles)
            
            # Add each converted reaction to results
            for converted_smiles in converted_reactions:
                successful_reactions.append({
                    'original_index': idx,
                    'original_smiles': original_smiles,
                    'converted_smiles': converted_smiles
                })
                
        except Exception as e:
            # Record failed reaction with detailed error info
            failed_reactions.append({
                'original_index': idx,
                'original_smiles': original_smiles,
                'error_type': type(e).__name__,
                'error_message': str(e),
                'traceback': traceback.format_exc()
            })
    
    # Save successful reactions
    if successful_reactions:
        success_df = pd.DataFrame(successful_reactions)
        success_filename = f'converted_reactions_{timestamp}.csv'
        success_df.to_csv(success_filename, index=False)
        print(f"\nSuccessfully converted reactions saved to: {success_filename}")
        print(f"Generated {len(successful_reactions)} reactions from {len(df)} original reactions")
    
    # Save failed reactions
    if failed_reactions:
        failed_df = pd.DataFrame(failed_reactions)
        failed_filename = f'failed_reactions_{timestamp}.csv'
        failed_df.to_csv(failed_filename, index=False)
        print(f"\nFailed reactions saved to: {failed_filename}")
        print(f"Number of failed reactions: {len(failed_reactions)}")
        
        # Group and display error types
        error_counts = failed_df['error_type'].value_counts()
        print("\nError type summary:")
        for error_type, count in error_counts.items():
            print(f"{error_type}: {count}")


if __name__ == "__main__":
    process_reactions_from_csv()

In [1]:
import itertools
from rdkit import Chem
import pandas as pd
import traceback
from tqdm import tqdm
from datetime import datetime

def generate_r_groups():
    """Generate R group replacements"""
    return ['C', 'CC', 'CCC']

def generate_ar_groups():
    """Generate Ar group replacements"""
    return ['c1ccccc1', 'c1ccc(C)cc1']  # 苯基和对甲苯基

def generate_y_groups():
    """Generate Y group replacements"""
    return ['O', 'S', 'N']  # NH will be handled separately for SMILES compatibility

def generate_ewg_groups():
    """Generate EWG group replacements"""
    return ['CN', 'COCC', 'C(=O)OCC']  # CN, COEt, COOEt

def generate_x_groups():
    """Generate X group replacements"""
    return ['Cl', 'Br']

def find_pattern_groups(smiles, pattern):
    """Find all unique pattern groups in the SMILES string"""
    import re
    patterns = re.findall(f'\\[{pattern}\\d*\\]', smiles)
    return sorted(set(patterns))

def is_catalyst_or_condition(molecule):
    """Check if a molecule string represents a catalyst or condition"""
    catalyst_patterns = ['[Pd', '[Pt', '[Rh', '[Ru', '[Fe', '[Cu', '[Ni', '[Au', '[Ag', '[Li']
    if molecule.startswith('[') and molecule.endswith(']'):
        return any(molecule.startswith(pat) for pat in catalyst_patterns)
    return False

def simplify_multistep_reaction(reaction_smiles):
    """Simplify a multistep reaction to single step"""
    steps = reaction_smiles.split('>')
    all_reactants = []
    
    for step in steps[:-1]:
        if step:
            molecules = step.split('.')
            for mol in molecules:
                if mol and not is_catalyst_or_condition(mol):
                    all_reactants.append(mol)
    
    final_products = steps[-1]
    seen = set()
    unique_reactants = []
    for r in all_reactants:
        if r not in seen:
            seen.add(r)
            unique_reactants.append(r)
    
    return f"{'.'.join(unique_reactants)}>>{final_products}"

def is_symmetric_equivalent(smiles1, smiles2):
    """Check if two SMILES strings represent the same molecule considering symmetry"""
    try:
        mol1 = Chem.MolFromSmiles(smiles1)
        mol2 = Chem.MolFromSmiles(smiles2)
        if mol1 is None or mol2 is None:
            return False
        return Chem.MolToInchiKey(mol1) == Chem.MolToInchiKey(mol2)
    except:
        return False

def process_x_replacements(smiles):
    """Process X group replacements including X2 and X3"""
    for i in range(3, 0, -1):  # Handle X3, X2, X1 in order
        pattern = f'[X{i}]' if i > 1 else '[X]'
        replacement = [f'[{x*i}]' for x in generate_x_groups()]
        if pattern in smiles:
            results = []
            for x_val in replacement:
                results.append(smiles.replace(pattern, x_val))
            return results
    return [smiles]

def process_halogen_replacements(smiles):
    """Process [F, Cl, Br, I] pattern into separate replacements"""
    # Match the exact pattern including spaces
    pattern = '[F, Cl, Br, I]'
    if pattern in smiles:
        results = []
        for halogen in ['F', 'Cl', 'Br', 'I']:
            # Create a new variant with this halogen
            new_smiles = smiles.replace(pattern, f'[{halogen}]')
            results.append(new_smiles)
        return results
    # Try without spaces pattern
    pattern_no_spaces = '[F,Cl,Br,I]'
    if pattern_no_spaces in smiles:
        results = []
        for halogen in ['F', 'Cl', 'Br', 'I']:
            # Create a new variant with this halogen
            new_smiles = smiles.replace(pattern_no_spaces, f'[{halogen}]')
            results.append(new_smiles)
        return results
    return [smiles]
# Add this new function for SMILES standardization
def standardize_smiles(smiles):
    """Standardize a SMILES string using RDKit"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except:
        return None

def convert_reaction_smiles(smiles):
    """Convert a generic reaction SMILES to specific reactions with all possible combinations"""
    smiles = simplify_multistep_reaction(smiles)
    reactants, products = smiles.split('>>')
    
    # Find all pattern groups
    r_patterns = find_pattern_groups(smiles, 'R')
    ar_patterns = find_pattern_groups(smiles, 'Ar')
    y_patterns = find_pattern_groups(smiles, 'Y')
    ewg_patterns = find_pattern_groups(smiles, 'EWG')
    
    # Generate all possible combinations
    r_groups = generate_r_groups()
    ar_groups = generate_ar_groups()
    y_groups = generate_y_groups()
    ewg_groups = generate_ewg_groups()
    halogens = ['F', 'Cl', 'Br', 'I']
    
    r_combinations = list(itertools.product(r_groups, repeat=len(r_patterns)))
    ar_combinations = list(itertools.product(ar_groups, repeat=len(ar_patterns)))
    y_combinations = list(itertools.product(y_groups, repeat=len(y_patterns)))
    ewg_combinations = list(itertools.product(ewg_groups, repeat=len(ewg_patterns)))
    
    unique_reactions = []
    
    all_combinations = itertools.product(
        r_combinations,
        ar_combinations,
        y_combinations,
        ewg_combinations,
        halogens
    )
    
    for combo in all_combinations:
        r_combo, ar_combo, y_combo, ewg_combo, halogen = combo
        current_reactants = reactants
        current_products = products
        
        # Replace halogen patterns
        for pattern in ['[F,Cl,Br,I]', '[F, Cl, Br, I]']:
            if pattern in current_reactants:
                current_reactants = current_reactants.replace(pattern, f'[{halogen}]')
            if pattern in current_products:
                current_products = current_products.replace(pattern, f'[{halogen}]')
        
        # Replace R groups
        for r_pattern, r_value in zip(r_patterns, r_combo):
            current_reactants = current_reactants.replace(r_pattern, r_value)
            current_products = current_products.replace(r_pattern, r_value)
        
        # Replace Ar groups
        for ar_pattern, ar_value in zip(ar_patterns, ar_combo):
            current_reactants = current_reactants.replace(ar_pattern, ar_value)
            current_products = current_products.replace(ar_pattern, ar_value)
        
        # Replace Y groups
        for y_pattern, y_value in zip(y_patterns, y_combo):
            y_replacement = '[NH]' if y_value == 'N' else f'[{y_value}]'
            current_reactants = current_reactants.replace(y_pattern, y_replacement)
            current_products = current_products.replace(y_pattern, y_replacement)
        
        # Replace EWG groups
        for ewg_pattern, ewg_value in zip(ewg_patterns, ewg_combo):
            current_reactants = current_reactants.replace(ewg_pattern, ewg_value)
            current_products = current_products.replace(ewg_pattern, ewg_value)
        
        # Process X replacements and standardize each molecule
        x_variants_reactants = process_x_replacements(current_reactants)
        for reactant_variant in x_variants_reactants:
            # Split reactants by '.' and standardize each molecule
            reactant_molecules = reactant_variant.split('.')
            standardized_reactants = []
            
            for molecule in reactant_molecules:
                std_mol = standardize_smiles(molecule)
                if std_mol is None:  # Skip if standardization fails
                    break
                standardized_reactants.append(std_mol)
            
            if len(standardized_reactants) != len(reactant_molecules):  # Skip if any molecule failed
                continue
                
            # Process and standardize products
            product_variant = process_x_replacements(current_products)[0]
            product_molecules = product_variant.split('.')
            standardized_products = []
            
            for molecule in product_molecules:
                std_mol = standardize_smiles(molecule)
                if std_mol is None:  # Skip if standardization fails
                    break
                standardized_products.append(std_mol)
                
            if len(standardized_products) != len(product_molecules):  # Skip if any molecule failed
                continue
            
            # Combine standardized molecules back into reaction SMILES
            std_reactants = '.'.join(standardized_reactants)
            std_products = '.'.join(standardized_products)
            full_reaction = f"{std_reactants}>>{std_products}"
            
            # Check for symmetry equivalence
            is_unique = True
            for existing_reaction in unique_reactions:
                existing_reactants, existing_products = existing_reaction.split('>>')
                if (is_symmetric_equivalent(std_reactants, existing_reactants) and 
                    is_symmetric_equivalent(std_products, existing_products)):
                    is_unique = False
                    break
            
            if is_unique:
                unique_reactions.append(full_reaction)
    
    return unique_reactions

def process_reactions_from_csv():
    """
    Process reactions from CSV file and save results
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    try:
        # Read input file without header, first column is SMILES
        df = pd.read_csv('name_reaction_smiles.csv', header=None)
        print(f"Successfully loaded CSV file with {len(df)} reactions")
    except Exception as e:
        print(f"Error reading input file: {str(e)}")
        return

    # Initialize results storage
    successful_reactions = []
    failed_reactions = []
    
    # Process each reaction
    for idx, row in tqdm(df.iterrows(), desc="Processing reactions"):
        try:
            original_smiles = str(row[0])  # Get SMILES from first column
            
            # Skip empty or invalid SMILES
            if pd.isna(original_smiles) or not original_smiles.strip():
                raise ValueError("Empty or invalid SMILES string")
            
            # Convert reaction
            converted_reactions = convert_reaction_smiles(original_smiles)
            
            # Add each converted reaction to results
            for converted_smiles in converted_reactions:
                successful_reactions.append({
                    'original_index': idx,
                    'original_smiles': original_smiles,
                    'converted_smiles': converted_smiles
                })
                
        except Exception as e:
            # Record failed reaction with detailed error info
            failed_reactions.append({
                'original_index': idx,
                'original_smiles': original_smiles,
                'error_type': type(e).__name__,
                'error_message': str(e),
                'traceback': traceback.format_exc()
            })
    
    # Save successful reactions
    if successful_reactions:
        success_df = pd.DataFrame(successful_reactions)
        success_filename = f'converted_reactions_{timestamp}.csv'
        success_df.to_csv(success_filename, index=False)
        print(f"\nSuccessfully converted reactions saved to: {success_filename}")
        print(f"Generated {len(successful_reactions)} reactions from {len(df)} original reactions")
    
    # Save failed reactions
    if failed_reactions:
        failed_df = pd.DataFrame(failed_reactions)
        failed_filename = f'failed_reactions_{timestamp}.csv'
        failed_df.to_csv(failed_filename, index=False)
        print(f"\nFailed reactions saved to: {failed_filename}")
        print(f"Number of failed reactions: {len(failed_reactions)}")
        
        # Group and display error types
        error_counts = failed_df['error_type'].value_counts()
        print("\nError type summary:")
        for error_type, count in error_counts.items():
            print(f"{error_type}: {count}")


if __name__ == "__main__":
    process_reactions_from_csv()

Successfully loaded CSV file with 1641 reactions


Processing reactions: 1641it [18:52:45, 41.42s/it]  



Successfully converted reactions saved to: converted_reactions_20241215_145624.csv
Generated 67117 reactions from 1641 original reactions


## 整合csv

In [ ]:
import pandas as pd

def clean_marked_bonds(df):
    # Remove rows where SMILES contains [U]
    return df[~df['SMILES'].str.contains('\[U\]', na=False)]

def process_files():
    # Read marked bonds files
    marked_bonds_apoc = pd.read_csv('./marked_bonds_apoc.csv')
    marked_bonds_dabook = pd.read_csv('./marked_bonds_DABOOK.csv')
    
    # Clean marked bonds data
    marked_bonds_apoc_clean = clean_marked_bonds(marked_bonds_apoc)
    marked_bonds_dabook_clean = clean_marked_bonds(marked_bonds_dabook)
    
    # Read processed smiles files
    processed_smiles_dabook = pd.read_csv('./processed_smiles_DABOOK_rev.csv')
    processed_smiles_apoc = pd.read_csv('./processed_smiles_apoc_rev.csv')
    
    # Combine marked bonds data
    marked_bonds_combined = pd.concat([
        marked_bonds_apoc_clean[['SMILES', 'Marked Bonds']],
        marked_bonds_dabook_clean[['SMILES', 'Marked Bonds']]
    ]).reset_index(drop=True)
    
    # Combine processed smiles data
    processed_smiles_combined = pd.concat([
        processed_smiles_dabook[['New_SMILES', 'New_Bonds']],
        processed_smiles_apoc[['New_SMILES', 'New_Bonds']]
    ]).reset_index(drop=True)
    
    # Rename columns for consistency
    processed_smiles_combined = processed_smiles_combined.rename(columns={
        'New_SMILES': 'SMILES',
        'New_Bonds': 'Bonds'
    })
    
    marked_bonds_combined = marked_bonds_combined.rename(columns={
        'Marked Bonds': 'Bonds'
    })
    
    # Combine all data
    final_data = pd.concat([
        marked_bonds_combined,
        processed_smiles_combined
    ]).reset_index(drop=True)
    
    # Remove duplicates
    final_data = final_data.drop_duplicates()
    
    # Save to CSV
    final_data.to_csv('combined_data.csv', index=False)
    
    return final_data

# Run the processing
result = process_files()

# Display first few rows and data info
print("\nFirst few rows of processed data:")
print(result.head())

print("\nDataset information:")
print(result.info())